# 01 — The Semantic Layer and Metric Views

**Time: about 15 minutes.**
Covers: Inconsistent metrics · Grain and fan-out · Unity Catalog metric views · `MEASURE()` · Synonyms and comments

### What you will be able to do afterwards

- Explain why two competent people get two different "revenue" numbers from the same tables.
- Define metrics once in a metric view (`CREATE VIEW ... WITH METRICS LANGUAGE YAML`) with joins, measures, dimensions, a global filter, synonyms and comments.
- Query it with `MEASURE()` and show that any breakdown re-aggregates to the same governed number.

In [ ]:
from helpers import utils, semantic_model

cfg = utils.use_lab_schema()
spark = utils.spark
print(f"Working in {cfg['catalog']}.{cfg['schema']}")

## Why raw tables produce inconsistent metrics

A metric is a **definition**, not a column: which rows count, at which grain, after which
exclusions, over which window. Raw tables hold the ingredients but not the recipe, so every
dashboard, notebook and AI assistant re-derives the recipe on its own, a little differently each time.

A **semantic layer** stores the recipe once, next to the data, and makes every consumer use it.
In Databricks that is a **Unity Catalog metric view**: a governed object whose YAML definition
separates *measures* (how to aggregate) from *dimensions* (what you may group and filter by).
The engine generates the correct SQL for whatever grouping the consumer asks for.

## Step 1 — Meet the data

In [ ]:
%sql
SELECT * FROM lab_orders LIMIT 5

In [ ]:
%sql
SELECT * FROM lab_refunds ORDER BY order_id LIMIT 8

## Step 2 — "What was our revenue in the last 90 days?"

Two analysts answer the CFO's question. Both queries look reasonable. Run them.

In [ ]:
%sql
-- Sales Ops dashboard: revenue = what customers paid, minus cancellations
SELECT ROUND(SUM(order_total), 2) AS revenue_90d
FROM lab_orders
WHERE status <> 'cancelled'
  AND order_date >= current_date() - INTERVAL 90 DAYS

In [ ]:
%sql
-- Finance notebook: revenue = order totals net of refunds
SELECT ROUND(SUM(o.order_total) - SUM(COALESCE(r.refund_amount, 0)), 2) AS revenue_90d
FROM lab_orders o
LEFT JOIN lab_refunds r ON o.order_id = r.order_id
WHERE o.status <> 'cancelled'
  AND o.order_date >= current_date() - INTERVAL 90 DAYS

Two numbers. They differ by tens of thousands of dollars, and the two analysts will now spend an
afternoon arguing about whose is right. Before taking sides, look closer.

In [ ]:
%sql
-- How many orders have more than one refund event? Each one is duplicated by the Finance join.
SELECT refund_events, COUNT(*) AS orders, ROUND(SUM(order_total), 2) AS order_total_duplicated
FROM (
  SELECT o.order_id, o.order_total, COUNT(r.refund_id) AS refund_events
  FROM lab_orders o JOIN lab_refunds r ON o.order_id = r.order_id
  GROUP BY o.order_id, o.order_total
)
GROUP BY refund_events
ORDER BY refund_events

In [ ]:
%sql
-- Two more things neither analyst filtered on
SELECT
  COUNT_IF(is_test_order)                                AS test_orders,
  ROUND(SUM(order_total) FILTER (WHERE is_test_order), 2) AS test_order_total,
  COUNT_IF(status = 'cancelled')                         AS cancelled_orders
FROM lab_orders
WHERE order_date >= current_date() - INTERVAL 90 DAYS

**What just happened**

- **Fan-out.** `lab_refunds` has one row per *refund event*. An order refunded in three instalments
  appears three times after the join, so Finance's `SUM(o.order_total)` counts its total three times.
  The refunds are subtracted correctly; the order totals are not.
- **Test orders.** Internal QA accounts place large orders every day. Neither query excludes them.
- **Gross vs. net.** Sales Ops never subtracts refunds at all.

So the argument is about a small gap between two numbers that are **both** wrong. The company
already wrote down what revenue means; nobody reads it:

In [ ]:
%sql
SELECT * FROM metric_glossary

## Step 3 — "How many active customers do we have?"

In [ ]:
%sql
-- Marketing: active = ordered in the last 30 days
SELECT COUNT(DISTINCT customer_id) AS active_customers
FROM lab_orders
WHERE order_date >= current_date() - INTERVAL 30 DAYS

In [ ]:
%sql
-- Customer Success: active = ordered in the last 90 days
SELECT COUNT(DISTINCT customer_id) AS active_customers
FROM lab_orders
WHERE order_date >= current_date() - INTERVAL 90 DAYS

Same story, bigger gap: about 30% of customers churn part-way through the data, so the
lookback window changes the answer by a thousand customers. Both teams are "right" by their own
definition. RevOps' definition (90 days, completed, non-test orders) is neither query.

## Step 4 — Fix the grain once

Before modelling, make refunds safe to join: one row per order. Every consumer that joins this
view instead of `lab_refunds` is immune to the fan-out.

In [ ]:
%sql
CREATE OR REPLACE VIEW order_refund_totals
COMMENT 'Refunds rolled up to ONE ROW PER ORDER. Join this, never lab_refunds, to orders.'
AS SELECT
  order_id,
  SUM(refund_amount) AS refund_amount,
  COUNT(*)           AS refund_events
FROM lab_refunds
GROUP BY order_id

## Step 5 — Build the metric view

| YAML key | What it does here |
|---|---|
| `version: 1.1` | Spec version. 1.1 is needed for `synonyms` and `display_name`. |
| `source` | The fact: `lab_orders`. |
| `filter` | Applied to **every** query: `NOT is_test_order`. |
| `joins` | Many-to-one lookups: `customer`, `product` and the order-grain `refunds` rollup. Joined only when a query needs them. |
| `dimensions` | What you can group and filter by. Newer docs call these `fields`; both keywords work. |
| `measures` | Aggregations, each with its exclusions built in (`FILTER (WHERE status <> 'cancelled')`). |
| `comment`, `synonyms`, `display_name` | Agent metadata. Genie reads them; `revenue` is a synonym of `net_revenue`, **not** of `gross_revenue`. |

The helper renders the YAML with your fully-qualified names and runs
`CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML AS $$ ... $$`. Read the YAML it prints.

In [ ]:
yaml_text = semantic_model.create_metric_view(show_yaml=True)

## Step 6 — Query it with `MEASURE()`

Measures must be wrapped in `MEASURE()`; dimensions are used like columns. `SELECT *` is not allowed.
Group by anything; the engine generates the joins and aggregation.

In [ ]:
%sql
-- Finance's breakdown, through the metric view: the refund join can no longer fan out
SELECT product_category, MEASURE(net_revenue) AS revenue_90d
FROM sales_metrics
WHERE order_date >= current_date() - INTERVAL 90 DAYS
GROUP BY product_category
ORDER BY revenue_90d DESC

In [ ]:
%sql
SELECT region, customer_segment,
       MEASURE(net_revenue)   AS net_revenue,
       MEASURE(gross_revenue) AS gross_bookings,
       MEASURE(order_count)   AS orders
FROM sales_metrics
WHERE order_date >= current_date() - INTERVAL 90 DAYS
GROUP BY ALL
ORDER BY net_revenue DESC

## Step 7 — Reconcile

`reconcile()` runs all four raw queries and four metric-view queries and saves every number to
`metric_reconciliation`. Each question is asked twice through the metric view, once as a total and
once as a breakdown that is summed back up. Through a semantic layer, both must give the same number.

In [ ]:
semantic_model.reconcile()
display(spark.table("metric_reconciliation").select("question", "approach", "value"))

Before: two raw answers per question, disagreeing with each other and with the glossary.
After: one number per question, however you slice it.

## Step 8 — What Genie will read

Everything you wrote into the YAML (comments, synonyms, display names) is stored in Unity Catalog
and returned by `DESCRIBE`. This is the context a Genie Agent and Genie Ontology work from in
Exercises 2–4.

In [ ]:
%sql
DESCRIBE TABLE EXTENDED sales_metrics AS JSON

---

## Challenge 1

Extend the metric view with:

| Name | Kind | Definition |
|---|---|---|
| `order_month` | dimension | `order_date` truncated to the calendar month |
| `refund_rate` | measure | refunds on completed orders ÷ gross bookings of completed orders, as a ratio between 0 and 1 |

Write the YAML entries in the cell below (same shape as the entries printed in Step 5; indentation is
normalised for you), run it, then run the verification query.

`create_metric_view` replaces the whole definition. If you re-run Step 5 later, re-run this cell too.

**TO DO**

In [ ]:
extra_dimensions = """
# TO DO: add a dimension named order_month
"""

extra_measures = """
# TO DO: add a measure named refund_rate
"""

semantic_model.create_metric_view(extra_dimensions=extra_dimensions, extra_measures=extra_measures)

In [ ]:
%sql
SELECT order_month,
       MEASURE(net_revenue) AS net_revenue,
       MEASURE(refund_rate) AS refund_rate
FROM sales_metrics
GROUP BY ALL
ORDER BY order_month

## Checks

In [ ]:
from helpers import test_runner
test_runner.run("01-semantic-layer-and-metric-views")